<a href="https://colab.research.google.com/github/MikanQuyt/BrainTumorDetect/blob/Nguyen_Thanh_Hai_tuan3/notebooks/week3_vit_lora_brisc2025_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
%cd /content
!rm -rf BrainTumorDetect

# Clone trực tiếp nhánh Tuần 3 đã vá toàn bộ lỗi
!git clone -b Nguyen_Thanh_Hai_tuan3 https://github.com/MikanQuyt/BrainTumorDetect.git
%cd BrainTumorDetect

# Gỡ torchao cũ của Colab & cài thư viện cần thiết
!pip uninstall -y torchao
!pip install -q timm transformers peft albumentations scikit-learn python-docx kaggle

/content
Cloning into 'BrainTumorDetect'...
remote: Enumerating objects: 777, done.
remote: Counting objects: 100% (230/230), done.
remote: Compressing objects: 100% (179/179), done.
remote: Total 777 (delta 71), reused 143 (delta 48), pack-reused 547 (from 1)
Receiving objects: 100% (777/777), 99.59 MiB | 17.73 MiB/s, done.
Resolving deltas: 100% (211/211), done.
/content/BrainTumorDetect
Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


In [2]:
# Thiết lập Kaggle API Key
!mkdir -p /root/.kaggle
!cp /content/kaggle.json /root/.kaggle/ 2>/dev/null || cp kaggle.json /root/.kaggle/ 2>/dev/null
!chmod 600 /root/.kaggle/kaggle.json

# Tải và giải nén BRISC 2025 (~30 giây)
!kaggle datasets download -d briscdataset/brisc2025 --unzip -p dataset/brisc2025

# Tiền xử lý CLAHE & Bounding Box (sinh ra 6,000 ảnh)
!python -m data.build_ds3 --root .

Dataset URL: https://www.kaggle.com/datasets/briscdataset/brisc2025
License(s): Attribution 4.0 International (CC BY 4.0)
100% 250M/250M [00:01<00:00, 176MB/s]

DS3 segmentation (có mask): 4793 mẫu
DS3 segmentation (có mask): 100% 4793/4793 [02:00<00:00, 39.76it/s]
DS3 classification (no_tumor): 1207 mẫu
DS3 classification (no_tumor): 100% 1207/1207 [00:16<00:00, 74.39it/s]

Đã xuất 6000 record (3646 nhóm bệnh nhân) -> dataset/ds3_annotations.json
  train: 4200 mẫu (70.00%), 2547 bệnh nhân | có mask=3355 | no_tumor=845, glioma=981, meningioma=1145, pituitary=1229
  val  :  900 mẫu (15.00%),  550 bệnh nhân | có mask= 719 | no_tumor=181, glioma=210, meningioma=245, pituitary=264
  test :  900 mẫu (15.00%),  549 bệnh nhân | có mask= 719 | no_tumor=181, glioma=210, meningioma=245, pituitary=264
Mẫu bị loại bỏ: 0 mẫu (tất cả hợp lệ)


In [3]:
from google.colab import drive
drive.mount('/content/drive')

# Nén và lưu toàn bộ 6,000 ảnh đã xử lý vào Google Drive của bạn
!zip -q -r /content/drive/MyDrive/ds3_processed.zip dataset/ds3_processed
print("✅ Đã lưu xong dữ liệu lên Google Drive!")

Mounted at /content/drive
✅ Đã lưu xong dữ liệu lên Google Drive!


In [4]:
!python -m experiments.run_lora_rank_ablation --summary-only

Config           |   r |    Trainable |        Total | Trainable % |  LoRA Only
------------------------------------------------------------------------------
linear_probe     |   0 |        3,076 |   85,801,732 |     0.0036% |          0
full_finetune    |   0 |   85,801,732 |   85,801,732 |   100.0000% |          0
vit_lora_r4      |   4 |      652,745 |   86,451,401 |     0.7550% |    147,456
vit_lora_r8      |   8 |      800,201 |   86,598,857 |     0.9240% |    294,912
vit_lora_r16     |  16 |    1,095,113 |   86,893,769 |     1.2603% |    589,824
vit_lora_r32     |  32 |    1,684,937 |   87,483,593 |     1.9260% |  1,179,648


In [5]:
!python -m engine.train_vit_lora --task binary --lora-r 8 --lora-alpha 16 --loss focal --label-smoothing 0.05 --epochs 10 --batch-size 32 --out runs/vit_lora_binary_r8

config.json: 100% 502/502 [00:00<00:00, 2.74MB/s]

model.safetensors: downloading bytes:  20% 70.7M/346M [00:00<00:02, 111MB/s, 3.34MB/s  ] 
model.safetensors: downloading bytes:  28% 97.5M/346M [00:01<00:01, 144MB/s, 6.87MB/s  ]
model.safetensors: reconstructing file:  28% 97.5M/346M [00:01<00:02, 93.1MB/s, 6.74MB/s  ]
model.safetensors: downloading bytes:  42% 145M/346M [00:01<00:01, 175MB/s, 11.7MB/s  ] 
model.safetensors: downloading bytes:  57% 197M/346M [00:01<00:00, 203MB/s, 15.7MB/s  ]
model.safetensors: downloading bytes:  66% 229M/346M [00:01<00:00, 223MB/s, 18.6MB/s  ]
model.safetensors: downloading bytes:  82% 285M/346M [00:01<00:00, 241MB/s, 23.5MB/s  ]
model.safetensors: downloading bytes:  91% 315M/346M [00:01<00:00, 254MB/s, 26.3MB/s  ]
model.safetensors: reconstructing file:  86% 296M/346M [00:01<00:00, 227MB/s, 22.8MB/s  ]
model.safetensors: downloading bytes: 100% 328M/328M [00:02<00:00, 153MB/s, 29.5MB/s  ]
model.safetensors: reconstructing file: 100% 346M/346M [00:

In [6]:
!python -m engine.train_vit_lora --task multiclass --lora-r 8 --lora-alpha 16 --loss focal --label-smoothing 0.05 --epochs 10 --batch-size 32 --out runs/vit_lora_multiclass_r8

Loading weights: 100% 198/198 [00:00<00:00, 1806.85it/s]
[transformers] ViTModel LOAD REPORT from: google/vit-base-patch16-224-in21k
Key                 | Status     |  | 
--------------------+------------+--+-
pooler.dense.bias   | UNEXPECTED |  | 
pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Mẫu: train=4200 val=900 test=900 | LoRA r=8, alpha=16 | Tham số khả huấn: 800,201/86,598,857 (0.92%)
  [E1 train] step 20/132 loss=1.7370 (cls=0.6978, box=1.0392)
  [E1 train] step 40/132 loss=1.5299 (cls=0.5869, box=0.9429)
  [E1 train] step 60/132 loss=1.3935 (cls=0.5027, box=0.8908)
  [E1 train] step 80/132 loss=1.2908 (cls=0.4453, box=0.8455)
  [E1 train] step 100/132 loss=1.2165 (cls=0.4022, box=0.8143)
  [E1 train] step 120/132 loss=1.1564 (cls=0.3735, box=0.7829)
  [E1 val] step 20/29 loss=0.8381 (cls=0.2165, box=0.6216)
Epoch 1/10  train_loss=1.1276 val_loss=0.7070 val_f1=0.9

In [7]:
!python -m experiments.run_lora_rank_ablation --summary-only

Config           |   r |    Trainable |        Total | Trainable % |  LoRA Only
------------------------------------------------------------------------------
linear_probe     |   0 |        3,076 |   85,801,732 |     0.0036% |          0
full_finetune    |   0 |   85,801,732 |   85,801,732 |   100.0000% |          0
vit_lora_r4      |   4 |      652,745 |   86,451,401 |     0.7550% |    147,456
vit_lora_r8      |   8 |      800,201 |   86,598,857 |     0.9240% |    294,912
vit_lora_r16     |  16 |    1,095,113 |   86,893,769 |     1.2603% |    589,824
vit_lora_r32     |  32 |    1,684,937 |   87,483,593 |     1.9260% |  1,179,648


In [8]:
!python reports/tuan3/build_week3_full_artifacts.py

/content/BrainTumorDetect/reports/tuan3/build_week3_full_artifacts.py:864: SyntaxWarning: invalid escape sequence '\i'
  - Thực hiện **Ablation Study số 1 (Khảo sát hạng LoRA)**: Chạy đối sánh thực nghiệm 4 cấu hình $r \in \{4, 8, 16, 32\}$ và so sánh trực tiếp với **Full Fine-tuning** và **Linear Probing** trên 3 tiêu chí: (1) Số lượng tham số khả huấn, (2) Thời gian hội tụ, và (3) F1-Score định lượng.
1. Lưu bảng số liệu JSON & CSV...
Tables saved to JSON and CSV.
2. Tạo các biểu đồ trực quan hóa độ phân giải cao...
Multiclass ablation figure generated.
Binary ablation figure generated.
Radar chart generated.
3. Tạo tài liệu Microsoft Word (.docx)...
Word document saved to:
 -> E:/archive/BRISC2025_LoRA_Experiments.docx
 -> E:/archive/BrainTumorDetect/reports/tuan3/BRISC2025_LoRA_Experiments.docx
 -> C:\Users\us\.gemini\antigravity\brain\a57d2bc4-ab77-4adb-814f-b7c153c70c2b/BRISC2025_LoRA_Experiments.docx
4. Tạo tài liệu Markdown khoa học...
Markdown reports saved successfully.

✅ Ho

In [11]:
!git restore reports/tuan1/
!git clean -fd

Removing "C:\\Users\\us\\.gemini\\antigravity\\brain\\a57d2bc4-ab77-4adb-814f-b7c153c70c2b/"
Removing E:/


In [12]:
# 1. Cấu hình thông tin tác giả
!git config user.email "hainguyen200507@gmail.com"
!git config user.name "hainguyen1107050705"

# 2. Thêm các báo cáo và biểu đồ mới nhất
!git add reports/tuan3/ image/tuan3/ notebooks/

# 3. Commit nội dung
!git commit -m "feat(colab): update week 3 training artifacts, reports and figures from Colab run"

# 4. Đẩy lên nhánh Tuần 3 (Thay YOUR_GITHUB_TOKEN bằng token của bạn)
# Nếu bạn chưa có token, xem Cách 2 ở dưới (đã đẩy sẵn từ máy tính lên rồi!)
!git push https://YOUR_GITHUB_TOKEN@github.com/MikanQuyt/BrainTumorDetect.git Nguyen_Thanh_Hai_tuan3

On branch Nguyen_Thanh_Hai_tuan3
Your branch is up to date with 'origin/Nguyen_Thanh_Hai_tuan3'.

nothing to commit, working tree clean
fatal: could not read Password for 'https://YOUR_GITHUB_TOKEN@github.com': No such device or address
